In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Specifică opțiuni

<details>
<summary><b>Versiuni de pachete</b></summary>

Codul de pe această pagină a fost dezvoltat folosind următoarele cerințe.
Recomandăm utilizarea acestor versiuni sau a unora mai noi.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Poți folosi opțiuni pentru a personaliza primitivele Estimator și Sampler. Această secțiune se concentrează pe cum să specifici opțiunile primitivelor Qiskit Runtime. Deși interfața metodei `run()` a primitivelor este comună tuturor implementărilor, opțiunile lor nu sunt. Consultă referințele API corespunzătoare pentru informații despre opțiunile [`qiskit.primitives`](https://docs.quantum.ibm.com/api/qiskit/primitives#primitives) și [`qiskit_aer.primitives`](https://qiskit.github.io/qiskit-aer/apidocs/aer_primitives.html).

Note despre specificarea opțiunilor în primitive:

- `SamplerV2` și `EstimatorV2` au clase de opțiuni separate. Poți vedea opțiunile disponibile și poți actualiza valorile opțiunilor în timpul sau după inițializarea primitivei.
- Folosește metoda `update()` pentru a aplica modificări atributului `options`.
- Dacă nu specifici o valoare pentru o opțiune, aceasta primește o valoare specială de `Unset` și se folosesc valorile implicite ale serverului.
- Atributul `options` este de tipul Python `dataclass`. Poți folosi metoda built-in `asdict` pentru a-l converti într-un dicționar.

<span id="pass-options"></span>
## Setează opțiunile primitivei
Poți seta opțiuni la inițializarea primitivei, după inițializarea primitivei sau în metoda `run()`. Consultă secțiunea [reguli de precedență](runtime-options-overview#options-precedence) pentru a înțelege ce se întâmplă când aceeași opțiune este specificată în mai multe locuri.

### Inițializarea primitivei
Poți transmite o instanță a clasei de opțiuni sau un dicționar la inițializarea unei primitive, care va face apoi o copie a acelor opțiuni. Astfel, modificarea dicționarului original sau a instanței de opțiuni nu afectează opțiunile deținute de primitive.

#### Clasa de opțiuni
La crearea unei instanțe a clasei `EstimatorV2` sau `SamplerV2`, poți transmite o instanță a clasei de opțiuni. Aceste opțiuni vor fi apoi aplicate când folosești `run()` pentru a efectua calculul. Specifică opțiunile în acest format: `options.option.sub-option.sub-sub-option = choice`. De exemplu: `options.dynamical_decoupling.enable = True`

Exemplu:

`SamplerV2` și `EstimatorV2` au clase de opțiuni separate ([`EstimatorOptions`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options) și [`SamplerOptions`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-sampler-options)).

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_ibm_runtime.options import EstimatorOptions

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

options = EstimatorOptions(
    resilience_level=2,
    resilience={"zne_mitigation": True, "zne": {"noise_factors": [1, 3, 5]}},
)

# or...
options = EstimatorOptions()
options.resilience_level = 2
options.resilience.zne_mitigation = True
options.resilience.zne.noise_factors = [1, 3, 5]

estimator = Estimator(mode=backend, options=options)

#### Dicționar
Poți specifica opțiunile ca dicționar la inițializarea primitivei.

In [2]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Setting options during primitive initialization
estimator = Estimator(
    backend,
    options={
        "resilience_level": 2,
        "resilience": {
            "zne_mitigation": True,
            "zne": {"noise_factors": [1, 3, 5]},
        },
    },
)

### Actualizarea opțiunilor după inițializare
Poți specifica opțiunile în acest format: `primitive.options.option.sub-option.sub-sub-option = choice` pentru a beneficia de auto-completare, sau folosește metoda `update()` pentru actualizări în masă.

Clasele de opțiuni `SamplerV2` și `EstimatorV2` ([`EstimatorOptions`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options) și [`SamplerOptions`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-sampler-options)) nu trebuie instanțiate dacă setezi opțiunile după inițializarea primitivei.

In [3]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = Estimator(mode=backend)

# Setting options after primitive initialization
# This uses auto-complete.
estimator.options.default_shots = 4000
# This does bulk update.
estimator.options.update(
    default_shots=4000, resilience={"zne_mitigation": True}
)

<span id="run-method"></span>
### Metoda Run()
Singurele valori pe care le poți transmite lui `run()` sunt cele definite în interfață. Adică, `shots` pentru Sampler și `precision` pentru Estimator. Aceasta suprascrie orice valoare setată pentru `default_shots` sau `default_precision` pentru rularea curentă.

In [4]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.circuit.library import random_iqp
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit1 = random_iqp(3)
circuit1.measure_all()
circuit2 = random_iqp(3)
circuit2.measure_all()

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

transpiled1 = pass_manager.run(circuit1)
transpiled2 = pass_manager.run(circuit2)

sampler = Sampler(mode=backend)
# Default shots to use if not specified in run()
sampler.options.default_shots = 500
# Sample two circuits at 128 shots each.
sampler.run([transpiled1, transpiled2], shots=128)

# Sample two circuits with different numbers of shots.
# 100 shots is used for transpiled1 and 200 for transpiled.
sampler.run([(transpiled1, None, 100), (transpiled2, None, 200)])

<RuntimeJobV2('d5k96cn853es738djikg', 'sampler')>

### Special cases

#### Resilience level (Estimator only)

The resilience level is not actually an option that directly impacts the primitive query, but specifies a base set of curated options to build off of. In general, level 0 turns off all error mitigation, level 1 turns on options for measurement error mitigation, and level 2 turns on options for gate and measurement error mitigation.

Any options you manually specify in addition to the resilience level are applied on top of the base set of options defined by the resilience level. Therefore, in principle, you could set the resilience level to 1, but then turn off measurement mitigation, although this is not advised.

In the following example, setting the resilience level to 0 initially turns off `zne_mitigation`, but `estimator.options.resilience.zne_mitigation = True` overrides the relevant setup from `estimator.options.resilience_level = 0`.

In [5]:
from qiskit_ibm_runtime import EstimatorV2, QiskitRuntimeService

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = EstimatorV2(backend)

estimator.options.default_shots = 100
estimator.options.resilience_level = 0
estimator.options.resilience.zne_mitigation = True

### Cazuri speciale
#### Nivelul de reziliență (doar Estimator)
Nivelul de reziliență nu este de fapt o opțiune care afectează direct interogarea primitivei, ci specifică un set de bază de opțiuni curate pe care să te construiești. În general, nivelul 0 dezactivează toate mitigările de erori, nivelul 1 activează opțiunile pentru mitigarea erorilor de măsurare, iar nivelul 2 activează opțiunile pentru mitigarea erorilor de Gate și de măsurare.

Orice opțiuni pe care le specifici manual în plus față de nivelul de reziliență sunt aplicate pe deasupra setului de bază de opțiuni definit de nivelul de reziliență. Prin urmare, în principiu, ai putea seta nivelul de reziliență la 1, dar să dezactivezi mitigarea măsurătorilor, deși acest lucru nu este recomandat.

În exemplul următor, setarea nivelului de reziliență la 0 dezactivează inițial `zne_mitigation`, dar `estimator.options.resilience.zne_mitigation = True` suprascrie configurarea relevantă din `estimator.options.resilience_level = 0`.

In [6]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.circuit.library import random_iqp
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit1 = random_iqp(3)
circuit1.measure_all()
circuit2 = random_iqp(3)
circuit2.measure_all()

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

transpiled1 = pass_manager.run(circuit1)
transpiled2 = pass_manager.run(circuit2)


# Setting shots during primitive initialization
sampler = Sampler(mode=backend, options={"default_shots": 4096})

# Setting options after primitive initialization
# This uses auto-complete.
sampler.options.default_shots = 2000

# This does bulk update.  The value for default_shots is overridden if you specify shots with run() or in the PUB.
sampler.options.update(
    default_shots=1024, dynamical_decoupling={"sequence_type": "XpXm"}
)

# Sample two circuits at 128 shots each.
sampler.run([transpiled1, transpiled2], shots=128)

<RuntimeJobV2('d5k96icjt3vs73ds5t0g', 'sampler')>

#### Shots (doar Sampler)
Metoda `SamplerV2.run` acceptă două argumente: o listă de PUB-uri, fiecare dintre ele putând specifica o valoare specifică PUB-ului pentru shots, și un argument de tip keyword shots. Aceste valori de shots fac parte din interfața de execuție a Sampler-ului și sunt independente de opțiunile Runtime Sampler. Ele au prioritate față de orice valori specificate ca opțiuni, pentru a respecta abstracția Sampler.

Cu toate acestea, dacă `shots` nu este specificat de niciun PUB sau în argumentul keyword al metodei run (sau dacă toate sunt `None`), atunci se folosește valoarea shots din opțiuni, în special `default_shots`.

Pe scurt, aceasta este ordinea de precedență pentru specificarea shots în Sampler, pentru orice PUB particular:

1. Dacă PUB-ul specifică shots, se folosește acea valoare.
2. Dacă argumentul keyword `shots` este specificat în `run`, se folosește acea valoare.
3. Dacă `num_randomizations` și `shots_per_randomization` sunt specificate ca opțiuni `twirling`, shots sunt produsul acelor valori.
3. Dacă `sampler.options.default_shots` este specificat, se folosește acea valoare.

Astfel, dacă shots sunt specificate în toate locurile posibile, se folosește cel cu cea mai mare prioritate (shots specificat în PUB).

#### Precizia (doar Estimator)
Precizia este analogă cu shots, descrisă în secțiunea anterioară, cu excepția faptului că opțiunile Estimator conțin atât `default_shots`, cât și `default_precision`. În plus, deoarece gate-twirling este activat implicit, produsul dintre `num_randomizations` și `shots_per_randomization` are prioritate față de aceste două opțiuni.

Concret, pentru orice PUB Estimator particular:

1. Dacă PUB-ul specifică precizia, se folosește acea valoare.
2. Dacă argumentul keyword precision este specificat în `run`, se folosește acea valoare.
2. Dacă `num_randomizations` și `shots_per_randomization` sunt specificate ca [opțiuni `twirling`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options) (activate implicit), se folosește produsul lor pentru a controla cantitatea de date.
3. Dacă `estimator.options.default_shots` este specificat, se folosește acea valoare pentru a controla cantitatea de date.
4. Dacă `estimator.options.default_precision` este specificat, se folosește acea valoare.

De exemplu, dacă precizia este specificată în toate patru locuri, se folosește cel cu cea mai mare prioritate (precizia specificată în PUB).

> **Note:** Precizia se scalează invers față de utilizare. Adică, cu cât precizia este mai mică, cu atât mai mult timp QPU este necesar pentru rulare.

## Opțiuni frecvent utilizate
Există multe opțiuni disponibile, dar următoarele sunt cele mai frecvent utilizate:

<span id="shots"></span>
### Shots
Pentru unii algoritmi, setarea unui număr specific de shots este o parte esențială a rutinelor lor. Shots (sau precizia) pot fi specificate în mai multe locuri. Acestea au prioritate după cum urmează:

Pentru orice Sampler PUB:

1. Shots cu valori întregi conținute în PUB
2. Valoarea `run(...,shots=val)`
3. Valoarea `options.default_shots`

Pentru orice Estimator PUB:

1. Precizia cu valori float conținută în PUB
2. Valoarea `run(...,precision=val)`
3. Valoarea `options.default_shots`
4. Valoarea `options.default_precision`

Exemplu:

In [7]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = Estimator(mode=backend)

estimator.options.max_execution_time = 2500

<span id="no-error-mitigation"></span>
## Turn off all error mitigation and error suppression

You can turn off all error mitigation and suppression if you are, for example, doing research on your own mitigation techniques. To accomplish this, for EstimatorV2, set `resilience_level = 0`. For SamplerV2, no changes are necessary because no error mitigation or suppression options are enabled by default.

Example:

Turn off all error mitigation and suppression in Estimator.

In [8]:
from qiskit_ibm_runtime import EstimatorV2 as Estimator, QiskitRuntimeService

# Define the service.  This allows you to access IBM QPU.
service = QiskitRuntimeService()

# Get a backend
backend = service.least_busy(operational=True, simulator=False)

# Define Estimator
estimator = Estimator(backend)

options = estimator.options

# Turn off all error mitigation and suppression
options.resilience_level = 0

### Timp maxim de execuție
Timpul maxim de execuție (`max_execution_time`) limitează cât timp poate rula un job. Dacă un job depășește această limită de timp, este anulat forțat. Această valoare se aplică job-urilor individuale, indiferent dacă sunt rulate în modul job, session sau batch.

Valoarea este setată în secunde, pe baza timpului cuantic (nu a timpului de ceas), care este cantitatea de timp pe care QPU o dedică procesării job-ului tău. Este ignorată când se utilizează modul de testare locală, deoarece acel mod nu folosește timp cuantic.